<a href="https://colab.research.google.com/github/subiksha0515/Recommendation_of_RAG/blob/main/Time_Weighted_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🏷️ Project Title: HR Leave Policy Retrieval System using Time-Weighted Semantic Search
---

---

📘 Project Overview
---

This project builds an intelligent document retrieval system over an HR Leave Policy PDF.
It combines semantic similarity search with a time-based ranking factor to ensure that the most recent and relevant policy rules are retrieved first when users ask leave-related questions.

The system also provides a 3D visualization to show how semantic score and time weight together determine the final ranking.


---

🧠 Definition — Time-Weighted Retrieval

Time-Weighted Retrieval is a technique where document ranking is based on:

Semantic similarity × Recency factor

This ensures that among similar documents, the latest or recently updated content is prioritized over older content.

---

🎯 What This Project Helps to Understand
---

This project demonstrates:

Why semantic search alone is not enough for rule-based documents

How document recency impacts correct decision making

How real HR/Legal/Medical systems prioritize updated policies

How to build a retrieval system that mimics real organizational document logic

How to visualize retrieval ranking using 3D analytics


---

🧩 Models and Techniques Used at Each Stage
---
| Stage         | Tool / Model       | Technique Used        | Purpose                              |
| ------------- | ------------------ | --------------------- | ------------------------------------ |
| PDF Reading   | `pypdf`            | Page extraction       | Treat document as structured content |
| Chunking      | Python logic       | Text chunking         | Create semantic units for embedding  |
| Embedding     | `all-MiniLM-L6-v2` | Sentence embeddings   | Capture meaning of chunks            |
| Vector DB     | `FAISS`            | Similarity search     | Find relevant chunks quickly         |
| Ranking Logic | Custom function    | Time-Weighted scoring | Prioritize recent content            |
| Visualization | `Plotly 3D`        | 3D ranking graph      | Show ranking logic visually          |
| UI            | `Gradio`           | Interactive interface | User query and response display      |


## 🧭 Step-by-Step Workflow
---

**Step 1 — Mount Drive.**
Access the HR_Leave_Policy PDF stored in Google Drive.

**Step 2 — Read PDF Pages.**
Use `pypdf` to extract text page by page.

**Step 3 — Assign Timestamps.**
Simulate old and new sections using different time values.

**Step 4 — Chunk the Text.**
Split each page into 300-word chunks for embedding.

**Step 5 — Create Embeddings.**
Use `SentenceTransformer (all-MiniLM-L6-v2)` to convert chunks into vectors.

**Step 6 — Store in FAISS.**
Save all embeddings in FAISS for fast similarity search.

**Step 7 — Apply Time-Weighted Retrieval.**
Rank results using: *Semantic Score × Time Weight*.

**Step 8 — Visualize & Interact.**


---


🔹 Cell 1 — Install

In [18]:
!pip install sentence-transformers faiss-cpu pypdf gradio plotly


🔹 Cell 2 — Mount Drive

In [24]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive



🔹 Cell 3 — Read Multiple PDFs with Timestamp

In [25]:
from pypdf import PdfReader
import time

pdf_path = "/content/drive/MyDrive/HR_Leave_Policy.pdf"
reader = PdfReader(pdf_path)

pages = []

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""

    # Simulate: first half old (2019), second half new (2025)
    if i < len(reader.pages)//2:
        fake_year = 2019
    else:
        fake_year = 2025

    timestamp = time.mktime(time.strptime(f"01 Jan {fake_year}", "%d %b %Y"))

    pages.append({
        "text": text,
        "page_id": i,
        "timestamp": timestamp
    })
print("Total Pages:", len(pages))


Total Pages: 39


🔹 Cell 4 — Chunking

In [26]:
def chunk_text(text, size=300):
    words = text.split()
    for i in range(0, len(words), size):
        yield " ".join(words[i:i+size])

chunks = []
for p in pages:
    for chunk in chunk_text(p["text"]):
        chunks.append({
            "chunk": chunk,
            "page_id": p["page_id"],
            "timestamp": p["timestamp"]
        })

len(chunks)


70

🔹 Cell 5 — Embeddings

In [27]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = [c["chunk"] for c in chunks]
embeddings = model.encode(texts).astype("float32")


🔹 Cell 6 — FAISS

In [28]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)



🔹 Cell 7 — Time Weight Function (Core)


In [29]:
def time_weight(ts):
    age_days = (time.time() - ts) / 86400
    return 1 / (1 + age_days)


🔹 Cell 8 — Time Weighted Retrieval

In [30]:
def retrieve_time_weighted(query, top_k=8):
    q_emb = model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)

    scored = []

    for i, idx in enumerate(indices[0]):
        sim_score = 1 / (1 + distances[0][i])
        t_weight = time_weight(chunks[idx]["timestamp"])
        final_score = sim_score * t_weight

        scored.append((final_score, chunks[idx], sim_score, t_weight))

    scored.sort(reverse=True)
    return scored[:5]


🔹 Cell 9 — 3D Visualization Function

In [37]:
import plotly.graph_objects as go
import time

def visualize_3d(scored_chunks):
    xs, ys, zs = [], [], []
    colors, hover_text = [], []

    for score, item, sim_score, t_weight in scored_chunks:
        xs.append(sim_score)
        ys.append(t_weight)
        zs.append(score)

        year = "2025" if item["timestamp"] > time.mktime(time.strptime("01 Jan 2022", "%d %b %Y")) else "2019"

        if year == "2025":
            colors.append("red")
            hover_text.append(f"New Policy (2025)<br>Semantic: {sim_score:.4f}<br>Time: {t_weight:.6f}<br>Final: {score:.4f}")
        else:
            colors.append("blue")
            hover_text.append(f"Old Policy (2019)<br>Semantic: {sim_score:.4f}<br>Time: {t_weight:.6f}<br>Final: {score:.4f}")

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=xs,
        y=ys,
        z=zs,
        mode='markers',  # ❗ no text here
        marker=dict(size=9, color=colors),
        hovertext=hover_text,
        hoverinfo='text'
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title='Semantic Score',
            yaxis_title='Time Weight',
            zaxis_title='Final Score'
        ),
        title="3D View: Semantic vs Time vs Final Ranking",
        annotations=[
            dict(
                text="🔴 Red → 2025 (New Policy) <br> 🔵 Blue → 2019 (Old Policy)",
                x=0,
                y=-0.1,
                xref="paper",
                yref="paper",
                showarrow=False,
                font=dict(size=14)
            )
        ]
    )

    return fig


🔹 Cell 10 — Gradio UI

In [38]:
import gradio as gr

def hr_bot_3d(query):
    results = retrieve_time_weighted(query)

    text_out = ""
    for score, item, sim, tw in results:
        text_out += f"""
### 📄 Chunk Result
- Semantic Score: {round(sim,4)}
- Time Weight: {round(tw,4)}
- Final Score: {round(score,4)}

{item['chunk'][:800]}...
"""

    fig = visualize_3d(results)

    return text_out, fig


gr.Interface(
    fn=hr_bot_3d,
    inputs=gr.Textbox(label="Ask HR Leave Question"),
    outputs=[
        gr.Markdown(label="Retrieved HR Policy Chunks"),
        gr.Plot(label="3D Ranking Visualization")
    ],
    title="⏱️ Time-Weighted HR Policy Retrieval with 3D Visualization"
).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c3cde219bdb1b7401a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Final Overview

This project presents a smart HR Leave Policy retrieval system that combines semantic search with a time-based ranking strategy. By integrating embeddings, FAISS vector search, and a time-weighted scoring mechanism, the system ensures that the most recent and relevant policy rules are retrieved when users ask leave-related questions. The addition of a 3D visualization further explains how ranking decisions are made internally.

✅ Project Conclusion

The project highlights an important limitation of traditional semantic search and solves it using Time-Weighted Retrieval. It demonstrates how real-world systems in HR, legal, and compliance domains prioritize updated documents for accurate decision-making. This approach results in a more reliable, context-aware, and industry-relevant document retrieval solution.